# SpaceX Falcon 9 Landing Prediction — Capstone

This notebook follows the capstone rubric in order. It combines the completed data collection, EDA, SQLite, Folium, Plotly Dash, and classification work into a presentation-ready narrative. The downloadable companion files include completed lab notebooks, an interactive map, and a runnable Dash app.

**Target:** predict whether a Falcon 9 first stage lands successfully (`Class = 1`).**Evidence:** IBM Skills Network course datasets; each module uses its own extract and row unit.**Caution:** the modeling sample is small (90 launches), so results are exploratory.


## 1. Data collection and data wrangling methodology

Launch data is collected from the SpaceX API snapshot, normalized from JSON, and enriched by resolving rocket, payload, launchpad, and core identifiers. The analysis filters to Falcon 9, single-core and single-payload launches; assigns a binary landing label; resets flight numbers; retains unknown landing-pad values; and fills missing payload mass with the Falcon 9 mean (6,104.96 kg).

The resulting EDA/modeling table contains 90 launches from 2010-06-04 through 2020-11-05. The SQL assignment uses a broader 101-row payload table, while the geospatial lab has 56 mapped launches.


In [ ]:
import pandas as pd
import numpy as np

EDA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
df = pd.read_csv(EDA_URL)
df["Date"] = pd.to_datetime(df["Date"])
df["Year"] = df["Date"].dt.year
print(df.shape, df["Class"].value_counts().sort_index().to_dict())


## 2. EDA and interactive visual analytics methodology

Use Pandas to inspect data quality and cohort sizes, then compare launch outcome with flight number, site, orbit, payload, and year. SQLite answers grouped and date-filtered questions on the payload-level extract. Folium clusters mapped launch outcomes and shows proximity estimates; a Dash dropdown filters the site-level, payload, and annual views.

Rates are paired with counts. Orbit categories with only a few launches are treated as descriptive rather than stable rankings.


In [ ]:
site_rates = df.groupby("LaunchSite", as_index=False).agg(launches=("Class", "size"), success_rate=("Class", "mean"))
orbit_rates = df.groupby("Orbit", as_index=False).agg(launches=("Class", "size"), success_rate=("Class", "mean"))
yearly_rates = df.groupby("Year", as_index=False).agg(launches=("Class", "size"), success_rate=("Class", "mean"))
site_rates.sort_values("success_rate", ascending=False)


## 3. Predictive analysis methodology

The target is `Class`. The 75 inputs are launch/booster context (`FlightNumber`, `PayloadMass`, `Orbit`, `LaunchSite`, `Flights`, `GridFins`, `Reused`, `Legs`, `Block`, `ReusedCount`, and `Serial`). Orbit, launch site, and serial are one-hot encoded. `LandingPad` is excluded because it may reflect the landing outcome rather than information available before launch.

Use a stratified 75/25 split (67 train, 23 test), five-fold stratified cross-validation on training data, and balanced accuracy for model selection because outcomes are imbalanced (60 successes, 30 unsuccessful landings). Compare regularized logistic regression, SVM, a shallow decision tree, and KNN.


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

input_cols = ["FlightNumber", "PayloadMass", "Orbit", "LaunchSite", "Flights", "GridFins", "Reused", "Legs", "Block", "ReusedCount", "Serial"]
X = pd.get_dummies(df[input_cols], columns=["Orbit", "LaunchSite", "Serial"], dtype=float).fillna(0).astype(float)
y = df["Class"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 4. EDA with visualization results

- Landing success is 60.0% at CCAFS SLC 40 (33/55), 77.3% at KSC LC-39A (17/22), and 76.9% at VAFB SLC-4E (10/13).
- The annual rate rises from 33.3% in 2014–15 to 84.2% in 2020, but varies between years.
- VLEO is 85.7% (12/14); GTO is 51.9% (14/27); SSO is 100% (5/5), which is too small to treat as a stable estimate.
- Payload distributions overlap across outcomes. Payload mass alone does not separate successful from unsuccessful landings.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sns.barplot(data=site_rates, x="success_rate", y="LaunchSite", color="#087E78", ax=axes[0])
axes[0].set(xlim=(0, 1), xlabel="Observed landing success rate", ylabel="Launch site", title="Success rate by site")
sns.lineplot(data=yearly_rates[yearly_rates.launches >= 5], x="Year", y="success_rate", marker="o", color="#087E78", ax=axes[1])
axes[1].set(ylim=(0, 1), xlabel="Year", ylabel="Observed landing success rate", title="Yearly success rate (n ≥ 5)")
plt.tight_layout()


## 5. EDA with SQL results

The completed SQL exercise uses SQLite to answer ten questions. Key results: four unique launch sites; NASA (CRS) payload total 48,213 kg; average F9 v1.1 payload 2,534.7 kg; first ground-pad landing success on 2015-12-22; and maximum payload 15,600 kg, tied by 12 booster versions. The SQL table has 101 payload rows, so its counts should not be merged with launch-level rates.


In [ ]:
import sqlite3
SQL_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
sql_df = pd.read_csv(SQL_URL).dropna(subset=["Date"])
con = sqlite3.connect(":memory:")
sql_df.to_sql("SPACEXTABLE", con, index=False, if_exists="replace")
queries = {
  "unique_sites": 'SELECT DISTINCT "Launch_Site" FROM SPACEXTABLE ORDER BY 1',
  "nasa_crs_payload_kg": 'SELECT SUM(CAST("PAYLOAD_MASS__KG_" AS REAL)) AS payload_kg FROM SPACEXTABLE WHERE "Customer" LIKE '%NASA (CRS)%'',
  "f9_v1_1_mean_payload_kg": 'SELECT AVG(CAST("PAYLOAD_MASS__KG_" AS REAL)) AS mean_kg FROM SPACEXTABLE WHERE "Booster_Version" LIKE 'F9 v1.1%'',
  "first_ground_pad_success": 'SELECT MIN("Date") AS date FROM SPACEXTABLE WHERE "Landing_Outcome" = 'Success (ground pad)'',
  "mission_outcomes": 'SELECT TRIM("Mission_Outcome") AS outcome, COUNT(*) AS n FROM SPACEXTABLE GROUP BY TRIM("Mission_Outcome") ORDER BY 1',
}
for name, query in queries.items():
    print(name)
    display(pd.read_sql_query(query, con))


## 6. Interactive map results using Folium

The Folium exercise maps 56 geocoded launch records at four coastal sites. Launch outcomes appear in green/red marker clusters, with site popups, layer controls, and a mouse-position readout. KSC LC-39A to the selected Atlantic coastline coordinate is approximately 7.43 km straight-line. The interactive map is included as `launch_site_map.html`; site rates in this 56-launch extract are descriptive.


In [ ]:
import folium
from folium.plugins import MarkerCluster, MousePosition
geo_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
geo = pd.read_csv(geo_url)
site_map = folium.Map(location=[31.2, -104], zoom_start=4, tiles="OpenStreetMap")
cluster = MarkerCluster(name="Launch outcomes").add_to(site_map)
for _, row in geo.iterrows():
    color = "green" if int(row["class"]) == 1 else "red"
    folium.Marker([row["Lat"], row["Long"]], popup=row["Launch Site"],
                  icon=folium.Icon(color="white", icon_color=color, icon="rocket", prefix="fa")).add_to(cluster)
MousePosition().add_to(site_map)
folium.LayerControl().add_to(site_map)
site_map


## 7. Plotly Dash dashboard results

The companion Dash app has a launch-site dropdown that updates three views: observed success by site, payload versus flight number and landing outcome, and annual success rate. Use the filter to inspect the cohort rather than reading one overall rate as universal.

Run `python spacex_dash_app.py` after installing `dash`, `plotly`, and `pandas` to open the interactive dashboard.


## 8. Predictive analysis (classification) results

The tuned shallow decision tree was the strongest model by training cross-validation balanced accuracy (79.4%). On the held-out 23 launches it reached 82.6% accuracy and 80.8% balanced accuracy, with 86.7% precision and recall. The confusion matrix is TN=6, FP=2, FN=2, TP=13.

The sample is small relative to the 75 predictors, and a random split does not test future-time generalization. Treat this model as a historical demonstration, not an operational guarantee.


In [ ]:
tree_grid = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=42),
    {"max_depth": [2, 4, 6, None], "min_samples_leaf": [1, 3, 5]},
    cv=cv, scoring="balanced_accuracy")
tree_grid.fit(X_train, y_train)
pred = tree_grid.predict(X_test)
print("best parameters:", tree_grid.best_params_)
print("cross-validation balanced accuracy:", tree_grid.best_score_)
print("holdout accuracy:", accuracy_score(y_test, pred))
print("holdout balanced accuracy:", balanced_accuracy_score(y_test, pred))
print("confusion matrix [actual rows, predicted columns]:
", confusion_matrix(y_test, pred))


## 9. Conclusion

Launch success improved over the observed history, site and orbit context matter, and payload mass alone is not a reliable rule. The classifier can support historical exploration, but it needs time-based and external validation before any operational use.


## 10. Creativity applied to enhance the presentation

The capstone pairs each outcome view with its sample count, connects geographic and mission-level exploration, and shows model errors as well as summary scores. The Folium layer controls and Dash site filter make the evidence explorable instead of leaving every comparison static.


## 11. Innovative insight

The three modules do not share the same observation unit: SQL has 101 payload rows, EDA/modeling has 90 launch rows, and Folium has 56 geocoded launches. The difference in row grain and cohort coverage can change apparent rates. Align the launch unit, site labels, and date window before pooling results or comparing module-level success rates.

### Sources

- [IBM Skills Network SpaceX EDA dataset](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv)
- [IBM Skills Network geospatial launch dataset](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv)
- [IBM Skills Network SQL SpaceX dataset](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv)
